# RF-DETR segmentation next to our semantic models: the bridge

*Tutorial: nothing here is exported to `src/` and `nbdev-test` does not run it. Every cell runs on CPU with synthetic data **except** the ones guarded by `RUN_RFDETR`, which need the optional `rfdetr` package in a **separate virtualenv** (the library never imports it and `pyproject.toml` does not list it).*

RF-DETR segmentation is an **instance** model: per image up to 100 objects, each with a box, a class, a score and a mask. Our library is **semantic**: one score per pixel and class. Research and decision: `docs/superpowers/specs/2026-11-22-rfdetr-seg-integration-research.md` (recommendation: run RF-DETR outside the library, compare on the same validation patches).

Two bridge functions make that comparison cheap:

| direction | function | module |
|---|---|---|
| our masks to RF-DETR training data | `write_coco_folder` (uses `mask_to_coco_instances`) | `masks` |
| RF-DETR output back to our metrics | `detections_to_semantic` | `inference` |

The flow: **patch cache -> COCO folder -> train RF-DETR outside -> reload -> detections -> semantic map -> our metrics / review report.**

In [ ]:
# setup: imports and the switch for the optional cells
import json, tempfile
from pathlib import Path
from types import SimpleNamespace

import cv2
import numpy as np
import torch

from segmentation_tools_pt.masks import mask_to_coco_instances, coco_instances_to_mask, write_coco_folder
from segmentation_tools_pt.inference import detections_to_semantic
from segmentation_tools_pt.metrics import ObjectMetrics, SegMetrics

RUN_RFDETR = False   # True only inside the separate rfdetr environment, with a GPU
tmp = Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)

## 1. A synthetic patch cache

`build_patch_dataset` (and `segpt-build`) write `patch_dir/{train,val}/images/{id}.png` and `masks/{id}.png`. We fake a tiny one with a thin line, a ring (a hole!) and a 2-px dot.

In [ ]:
def fake_patch(i):
    img = (rng.normal(60, 5, (96, 96))).clip(0, 255).astype(np.uint8)
    m = np.zeros((96, 96), np.uint8)
    cv2.line(m, (5, 10 + 8 * i), (90, 40 + 8 * i), 255, 1)       # 1-px line
    cv2.circle(m, (30, 70), 12, 255, 3)                         # ring: has a hole
    m[20:22, 70:72] = 255                                       # 2-px dot
    img[m > 0] = 200
    return img, m

for split, n in (('train', 6), ('val', 2)):
    for sub in ('images', 'masks'):
        (tmp / 'patches' / split / sub).mkdir(parents=True)
    for i in range(n):
        img, m = fake_patch(i)
        cv2.imwrite(str(tmp / 'patches' / split / 'images' / f'p{i}.png'), img)
        cv2.imwrite(str(tmp / 'patches' / split / 'masks' / f'p{i}.png'), m)
print(sorted(p.name for p in (tmp / 'patches' / 'train' / 'images').iterdir()))

## 2. Patch cache to COCO folder

`write_coco_folder` writes `train/` and `valid/` (the names RF-DETR looks for), 8-bit PNGs, and one `_annotations.coco.json` per folder. A single class name means a binary mask; several names mean label values 1..K. By default components with holes are written as RLE and the rest as polygons (`segmentation='auto'`).

In [ ]:
summary = write_coco_folder(str(tmp / 'patches'), str(tmp / 'coco'), class_names=['defect'])
print(summary)                                   # {folder: (n_images, n_annotations)}
coco = json.loads((tmp / 'coco' / 'train' / '_annotations.coco.json').read_text())
print(sorted(p.name for p in (tmp / 'coco' / 'train').iterdir())[:3], '...')
kinds = [type(a['segmentation']).__name__ for a in coco['annotations']]
print({k: kinds.count(k) for k in set(kinds)}, '(dict = RLE, list = polygons)')

### What the mask-to-COCO step loses

Measured in `nbs/02_masks.ipynb` (64x64 synthetic masks, 8-connectivity). Default `epsilon=0` is lossless on the pixel grid:

| case | `auto` (default) | `polygon` | `rle` |
|---|---|---|---|
| 1-px thin line (diagonal / horizontal) | IoU 1.0000 | 1.0000 | 1.0000 |
| square with a hole | 1.0000 (RLE) | **0.8889**, hole filled | 1.0000 |
| 2-px and 1-px dots | 1.0000 | 1.0000 | 1.0000 |
| island inside a hole | 1.0000 | **0.9339** | 1.0000 |

`epsilon > 0` simplifies polygons: on a radius-20 disc 0.5 px gives 104 instead of 552 numbers at IoU 0.975, 1 px gives 32 numbers at IoU 0.950. Do not simplify masks with 1-3 px defects. **Not measured:** how RF-DETR's own loader (resize to 312-768 px, augmentations) treats a 1-px feature; that is the benchmark ticket's job.

In [ ]:
# verify the round trip on this tutorial's own masks
for i in range(6):
    gt = cv2.imread(str(tmp / 'patches' / 'train' / 'masks' / f'p{i}.png'), cv2.IMREAD_UNCHANGED) > 0
    anns = mask_to_coco_instances(gt.astype(np.uint8))
    back = coco_instances_to_mask(anns, gt.shape) > 0
    iou = (back & gt).sum() / (back | gt).sum()
    assert iou >= 0.99, (i, iou)
print('round-trip IoU >= 0.99 on all', 6, 'patches')

## 3. Train RF-DETR outside the library

Run this in its own environment (`pip install "rfdetr[train]"`; at the time of writing the `[train]` extra needed `--prerelease=allow`). Weights download to `~/.roboflow/models/`; on an air-gapped node stage them first.

Three gotchas from the research and our smoke runs, all about the same toy 1-channel Nano model:

1. **Do not call `predict` on the object you just trained** (1-channel model): it raised `Expected 3 but got 1`. Save, then reload with `RFDETR.from_checkpoint(path)` and predict from that.
2. **On small data prefer `last_ema.pth` over `checkpoint_best_total.pth`.** The "best" rule is rank-based (mask mAP), so it picked epoch 1, whose scores were about 0.03 (nothing passes a 0.3 threshold), while `last_ema.pth` from the same run scored about 0.93. Check on real data before trusting either.
3. **ONNX export is fixed batch 1 and fixed size** (`input [1, C, 312, 312]` for Nano; batch 2 or 384 px fails in onnxruntime). Outputs are `dets`, `labels`, `masks`, not our `logits (B, C, H, W)`, so `export_onnx` conventions do not apply.

In [ ]:
if RUN_RFDETR:   # optional: needs rfdetr + GPU, never run by the library tests
    from rfdetr import RFDETRSegNano
    model = RFDETRSegNano(num_channels=1)                     # our patches are single-channel
    model.train(dataset_dir=str(tmp / 'coco'), epochs=60, batch_size=4, lr=3e-4, output_dir=str(tmp / 'rf_run'))
else:
    print('skipped: set RUN_RFDETR = True in the rfdetr environment')

In [ ]:
if RUN_RFDETR:   # optional
    from rfdetr import RFDETR
    ckpt = tmp / 'rf_run' / 'last_ema.pth'                    # gotcha 2: not checkpoint_best_total.pth
    model = RFDETR.from_checkpoint(str(ckpt))                 # gotcha 1: reload, do not reuse the trained object
    val_img = cv2.imread(str(tmp / 'patches' / 'val' / 'images' / 'p0.png'), cv2.IMREAD_UNCHANGED)
    dets = model.predict(val_img, threshold=0.3)              # supervision.Detections: .mask .class_id .confidence

## 4. Detections back to a semantic map

`detections_to_semantic` takes anything with `.mask (N, H, W) bool`, `.class_id (N,)`, `.confidence (N,)` (a `supervision.Detections` qualifies) or a plain `(mask, class_id, confidence)` tuple. Where instances overlap the **higher score wins**. Here we fake RF-DETR output by cutting two instances out of a ground-truth mask, plus a false alarm.

In [ ]:
gt = cv2.imread(str(tmp / 'patches' / 'val' / 'masks' / 'p0.png'), cv2.IMREAD_UNCHANGED) > 0
anns = mask_to_coco_instances(gt.astype(np.uint8))
inst = np.stack([coco_instances_to_mask([a], gt.shape) > 0 for a in anns])      # (N, H, W) bool, one per component
fa = np.zeros_like(gt); fa[80:84, 80:84] = True                                  # a false alarm
fake = SimpleNamespace(mask=np.concatenate([inst, fa[None]]), class_id=np.ones(len(inst) + 1, int),
                       confidence=np.array([0.95] * len(inst) + [0.4]))
sem = detections_to_semantic(fake, class_ids=[1])                  # (H, W) long, 0 = background
sem_hi = detections_to_semantic(fake, class_ids=[1], threshold=0.5)  # drop the low-score false alarm
print(sem.shape, sem.dtype, 'fg px:', int((sem > 0).sum()), '| after threshold 0.5:', int((sem_hi > 0).sum()))

## 5. Compare with our metrics

`ObjectMetrics` takes binary maps; `SegMetrics` takes logits, so turn the `output='stack'` scores into logits. The same two calls score a semantic model's prediction, which is what makes the side-by-side fair (same validation patches, same metric code). The same arrays also drop into `review_report.Sample(pred=...)`.

In [ ]:
for name, s in (('no threshold', sem), ('threshold 0.5', sem_hi)):
    om = ObjectMetrics(); om.update((s == 1).astype(np.uint8), gt.astype(np.uint8))
    r = om.compute(); print(f"{name:14s} detection_rate={r['detection_rate']:.2f} false_alarms={r['false_alarms']}")

stack = detections_to_semantic(fake, class_ids=[1], output='stack')               # (1, H, W) float scores
sm = SegMetrics(num_classes=1)
sm.update(torch.logit(torch.from_numpy(stack).clamp(1e-4, 1 - 1e-4))[None], torch.from_numpy(gt.astype(np.float32))[None])
print('pixel IoU of the instance prediction:', round(sm.compute()['iou'], 3))

## Next

- Real data: run section 3 on your own patch cache, then repeat sections 4-5 on the **same validation patches** as the semantic model and compare `detection_rate` at equal false alarms (the decision rule in the research spec, section 11).
- Large images: RF-DETR has no tiling; cut tiles yourself and decide how to merge instances cut at tile borders. That is the follow-up work gated on this comparison.
- Skills: `choose-model-architecture` (when to use an instance model) and `run-inference` (`detections_to_semantic`). Reference: `docs/TRAINING_CLI.md`, section "RF-DETR instance models".